In [2]:
import os
import time
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [ ]:
BASE_DIR = "../data/docs"

model = "llama-3.3-70b-versatile"

In [5]:

llm = ChatGroq(
    model = model,
    temperature=0.2
)

In [6]:
TOPICS = {
    "billing": [
        "Invoice not generated",
        "Card charged twice",
        "Payment failed but amount deducted",
        "Update billing address",
        "Download past invoices",
        "GST invoice request",
        "International payment failed",
        "Billing currency mismatch",
        "Payment stuck in pending",
        "Change payment method",
        "Unauthorized charge",
        "Bank transfer failed",
        "UPI payment issues",
        "Tax calculation incorrect",
        "Billing email not received",
        "Invoice shows wrong company name",
        "Payment gateway timeout",
        "Advance billing issue",
        "Late fee charged",
        "Billing cycle confusion"
    ],
    "login": [
        "Forgot password",
        "Account locked after failed attempts",
        "2FA not working",
        "Login from new device blocked",
        "Email verification failed",
        "Password reset email not received",
        "SSO login failed",
        "Captcha not loading",
        "Login page stuck loading",
        "Account suspended message",
        "Wrong credentials error",
        "Session expired repeatedly",
        "Login works on mobile but not desktop",
        "IP blocked during login",
        "Security challenge failed",
        "Login redirect loop",
        "Browser compatibility issue",
        "Login after password change failed",
        "Account access revoked",
        "Login latency issue"
    ],
    "subscription": [
        "Upgrade subscription plan",
        "Downgrade subscription plan",
        "Cancel subscription",
        "Pause subscription",
        "Trial expired",
        "Extend free trial",
        "Subscription not activated",
        "Plan features not reflected",
        "Auto-renewal failed",
        "Change billing cycle",
        "Subscription renewal failed",
        "Multiple subscriptions active",
        "Subscription charged after cancellation",
        "Re-activate canceled plan",
        "Subscription status incorrect",
        "Upgrade payment failed",
        "Downgrade effective date confusion",
        "Plan limits exceeded",
        "Subscription not visible",
        "Enterprise plan inquiry"
    ],
    "refunds": [
        "Refund eligibility",
        "Refund status",
        "Refund delayed beyond timeline",
        "Partial refund request",
        "Refund failed",
        "Refund to wrong account",
        "Chargeback initiated",
        "Refund after trial cancellation",
        "Duplicate payment refund",
        "Refund declined",
        "Refund processing time",
        "Refund policy clarification",
        "Refund after downgrade",
        "Refund receipt request",
        "Refund reversed",
        "Refund not reflected in bank",
        "Refund for unused credits",
        "Refund on annual plan",
        "Refund dispute",
        "Refund after account suspension"
    ]
}


In [7]:
MD_TEMPLATE = """# {title}

## Category
{category}

## Problem
{problem}

## Solution
{solution}

## Notes
{notes}

## Keywords
{keywords}
"""


In [9]:
from langchain_core.messages import HumanMessage

In [10]:
def generate_faqs(category : str, title: str) -> str:
    prompt = f"""
    You are writing SaaS customer support documentation.

    Write a single help-center article using this exact structure:

    Problem:
    (one clear customer problem statement)

    Solution:
    (step-by-step resolution, numbered)

    Notes:
    (edge cases, limitations, warnings)

    Keywords:
    (5-7 comma-separated search keywords)

    Topic : {title}
    Category : {category}

    Rules:
    - No marketing language
    - Clear, factual tone
    - No hallucinated policies
    - Keep it realistic for a SaaS product
    """


    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()

    sections = {}
    current = None

    for line in content.splitlines():
        line = line.strip()
        if line.endswith(":"):
            current = line[:-1].lower()
            sections[current] = []
        elif current:
            sections[current].append(line)

    return MD_TEMPLATE.format(
        title = title,
        category = category,
        problem = " ".join(sections.get("problem", ["N/A"])),
        solution = "\n".join(sections.get("solution", ["N/A"])),
        notes = " ".join(sections.get("notes", ["N/A"])),
        keywords = ", ".join(sections.get("keywords", ["support"]))
    )

In [11]:
def main():
    os.makedirs(BASE_DIR, exist_ok=True)
    total = 0

    for category, titles in TOPICS.items():
        category_dir = os.path.join(BASE_DIR,category)
        os.makedirs(category_dir, exist_ok=True)

        for title in titles:
            filename = title.lower().replace(" ", "_") + ".md"
            path = os.path.join(category_dir, filename)

            print(f"Generating -> {category}/{filename}")

            md = generate_faqs(category, title)

            with open(path, "w", encoding="utf-8") as f:
                f.write(md)

            total +=1
            time.sleep(0.4)

    print(f"\n Generated {total} FAQ markdown documents")

In [12]:
if __name__ == "__main__":
    main()

Generating -> billing/invoice_not_generated.md
Generating -> billing/card_charged_twice.md
Generating -> billing/payment_failed_but_amount_deducted.md
Generating -> billing/update_billing_address.md
Generating -> billing/download_past_invoices.md
Generating -> billing/gst_invoice_request.md
Generating -> billing/international_payment_failed.md
Generating -> billing/billing_currency_mismatch.md
Generating -> billing/payment_stuck_in_pending.md
Generating -> billing/change_payment_method.md
Generating -> billing/unauthorized_charge.md
Generating -> billing/bank_transfer_failed.md
Generating -> billing/upi_payment_issues.md
Generating -> billing/tax_calculation_incorrect.md
Generating -> billing/billing_email_not_received.md
Generating -> billing/invoice_shows_wrong_company_name.md
Generating -> billing/payment_gateway_timeout.md
Generating -> billing/advance_billing_issue.md
Generating -> billing/late_fee_charged.md
Generating -> billing/billing_cycle_confusion.md
Generating -> login/for